## Imports


First install the required packages using pip.
Run the following cell to install the packages listed in the `requirements.txt` file.


In [ ]:
# %pip install -r requirements.txt

Import the necessary libraries that will be used throughout the project.


In [ ]:
import numpy as np  # For working with large arrays and matrices
import matplotlib.pyplot as plt  # For plotting and visualization
import torch  # For deep learning and tensor computations
import monai  # For medical imaging AI tools and utilities
import sklearn  # For machine learning algorithms and utilities
import scipy  # For scientific computing and image processing
import nibabel as nib  # For loading and working with neuro-imaging data
import os  # For interacting with the operating system
import shutil  # For high-level file operations
import time  # For measuring execution time
import joblib  # For parallel processing
import tqdm  # For progress bars
import json  # For working with JSON data
import random  # For random number generation and shuffling
import math  # For mathematical functions and operations

## Task 1: Understand the Dataset


The training dataset provided for the BraTS21 consists of 1,251 brain MRI scans along with segmentation annotations of tumorous regions. The 3D volumes were skull-stripped and resampled to 1 mm isotropic resolution, with dimensions of (240, 240, 155) voxels.

In the **BraTS 2021 (Brain Tumor Segmentation)** dataset, the training data is organized into subject-specific folders. Each folder typically contains **5 NIfTI files** (`.nii.gz`), representing four different MRI modalities and one ground truth segmentation mask.

---

### 1. `_t1.nii.gz` (Native T1-weighted)

The native T1-weighted scan is the "baseline" structural image. It is excellent for viewing normal brain anatomy.

### 2. `_t1ce.nii.gz` (Post-contrast T1-weighted)

Also known as **T1Gd** (Gadolinium-enhanced), this is a T1-weighted scan taken after a contrast agent (Gadolinium) is injected into the patient.

### 3. `_t2.nii.gz` (T2-weighted)

T2-weighted scans are highly sensitive to water and fluid.

### 4. `_flair.nii.gz` (Fluid Attenuated Inversion Recovery)

FLAIR is a special T2-weighted sequence where the signal from free-flowing water (like CSF) is suppressed or "nullified."

### 5. `_seg.nii.gz` (Ground Truth Segmentation)

This is the manually annotated label file created by expert neuroradiologists. It contains integer values (labels) that categorize each voxel into a specific tumor sub-region:

| Label | Sub-region  | Description                                        |
| :---- | :---------- | :------------------------------------------------- |
| **0** | Background  | Healthy brain tissue or non-brain space.           |
| **1** | **NCR/NET** | Necrotic (dead) and Non-Enhancing tumor core.      |
| **2** | **ED**      | Peritumoral Edema (swelling around the tumor).     |
| **4** | **ET**      | GD-Enhancing Tumor (the active part of the tumor). |

---

### Summary of Tumor Sub-regions

When evaluating models, these labels are usually grouped into three nested classes:

- **Whole Tumor (WT):** Labels 1 + 2 + 4 (Everything abnormal)
- **Tumor Core (TC):** Labels 1 + 4 (The solid tumor)
- **Enhancing Tumor (ET):** Label 4 only


In [ ]:
# Modality names in the BraTS dataset and their corresponding file suffixes
modalities = ["t1", "t1ce", "t2", "flair"]

# Load the first subject's data (BraTS2021_00000) for visualization

imgs = [
    nib.load(f".raw/BraTS2021_00000/BraTS2021_00000_{mod}.nii.gz")
    .get_fdata()
    .astype(np.float32)
    for mod in modalities
]
lbl = (
    nib.load(".raw/BraTS2021_00000/BraTS2021_00000_seg.nii.gz")
    .get_fdata()
    .astype(np.uint8)
)

print("Shapes of the loaded images and label:")
for i, img in enumerate(imgs):
    print(f"Image - {modalities[i]} shape: {img.shape}")

print(f"Label shape: {lbl.shape}")

fig, axes = plt.subplots(nrows=1, ncols=5, figsize=(15, 15))

for i, img in enumerate(imgs):
    axes[i].imshow(img[:, :, 75], cmap="gray")

axes[-1].imshow(lbl[:, :, 75], vmin=0, vmax=4)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Task 2: Preprocessing & Data Augmentation


### Preprocess MRI scans


In [ ]:
# Data directories for raw and processed data
raw_data_path = ".raw"
processed_data_path = ".processed"

# Check if the raw data directory exists else raise an error
if not os.path.exists(raw_data_path):
    raise FileNotFoundError(f"Raw data directory '{raw_data_path}' does not exist.")

# Create the processed data directory if it doesn't exist
if not os.path.exists(processed_data_path):
    os.makedirs(processed_data_path)

# List all subject directories in raw and processed data paths
raw_data_dirs = [
    d
    for d in os.listdir(raw_data_path)
    if os.path.isdir(os.path.join(raw_data_path, d))
]
processed_data_dirs = [
    d
    for d in os.listdir(processed_data_path)
    if os.path.isdir(os.path.join(processed_data_path, d))
]

# If incomplete processing is detected, clear the processed data directory
if len(raw_data_dirs) != len(processed_data_dirs):
    shutil.rmtree(processed_data_path)
    os.makedirs(processed_data_path)

# If all raw data has already been processed, ask the user if they want to reprocess the data
if len(raw_data_dirs) == len(processed_data_dirs):
    print("Total data samples:", len(raw_data_dirs))
    print("All raw data has already been processed.")
    print("Do you want to reprocess the data? (y/n)")
    if input().lower() == "y":
        shutil.rmtree(processed_data_path)
        os.makedirs(processed_data_path)


In [ ]:
# Function to run a given function in parallel across multiple IDs using joblib
def run_parallel(func, ids):
    return joblib.Parallel(n_jobs=os.cpu_count())(
        joblib.delayed(func)(id) for id in ids
    )

In [ ]:
def collect_intensities(ids):

    intensities_dicts = run_parallel(get_intensities, ids)

    intensities = {}
    intensity_min = {}
    intensity_max = {}
    intensity_mean = {}
    intensity_std = {}

    for mod in modalities:
        # Chain the intensity values from all patients for the current modality into a single list.
        all_values = []
        for d in intensities_dicts:
            all_values.extend(d[mod])
        intensities[mod] = all_values

        # Calculate the 0.5th and 99.5th percentiles to determine the minimum and maximum intensity values for normalization,
        # and calculate the mean and standard deviation for further normalization steps.
        intensity_min[mod] = np.percentile(intensities[mod], [0.5, 99.5])[0]
        intensity_max[mod] = np.percentile(intensities[mod], [0.5, 99.5])[1]
        intensity_mean[mod] = np.mean(intensities[mod])
        intensity_std[mod] = np.std(intensities[mod])

    return {
        "min": intensity_min,
        "max": intensity_max,
        "mean": intensity_mean,
        "std": intensity_std,
    }


def get_intensities(id):
    intensity = {}

    label = (
        nib.load(
            os.path.join(
                raw_data_path,
                f"BraTS2021_{id}",
                f"BraTS2021_{id}_seg.nii.gz",
            )
        )
        .get_fdata()
        .astype(np.uint8)
    )

    for mod in modalities:
        image = (
            nib.load(
                os.path.join(
                    raw_data_path,
                    f"BraTS2021_{id}",
                    f"BraTS2021_{id}_{mod}.nii.gz",
                )
            )
            .get_fdata()
            .astype(np.float32)
        )
        # Extract brain region
        foreground_area = np.where(label > 0)
        # Store the intensity values of the brain region for the current modality in the intensity dictionary as a list.
        intensity[mod] = image[foreground_area].tolist()
    return intensity

In [ ]:
# Function to preprocess images for a list of IDs in parallel
def preprocess_images(ids):
    run_parallel(process_image, ids)


def process_image(id):
    # Create a directory for the processed data of the current patient
    os.makedirs(os.path.join(processed_data_path, f"BraTS2021_{id}"))

    # Load the image for the current modality and patient
    images = {}
    for mod in modalities:
        image = nib.load(
            os.path.join(
                raw_data_path,
                f"BraTS2021_{id}",
                f"BraTS2021_{id}_{mod}.nii.gz",
            )
        )
        images[mod] = image

    # Save the header and affine of the T1 image to use for the processed image.
    # All modalities have the same header and affine, so we can use the T1 image as a reference.
    img_header = images["t1"].header
    img_affine = images["t1"].affine

    ## Normalize the images
    # Step 1: Clip the intensity values within the minimum and maximum values.
    # Step 2: Normalize the intensity values using the mean and standard deviation.
    normalized_data = {}
    for mod in modalities:
        image = images[mod].get_fdata().astype(np.float32)
        image = np.clip(image, intensity_stats["min"][mod], intensity_stats["max"][mod])
        image = (image - intensity_stats["mean"][mod]) / intensity_stats["std"][mod]
        normalized_data[mod] = image

    # Stack the normalized images of all modalities into a single 4D array with shape (4, 240, 240, 155).
    channels_data = np.stack([normalized_data[mod] for mod in modalities], axis=0)

    ## Padding the images
    # Padding is done to ensure that all images have a shape which is divisible by 32.
    # This is important for the U-Net architecture which down-samples the images by a factor of 32 in the encoder.
    # Target size: (256, 256, 160)
    # Original size: (240, 240, 155)
    # Difference: Height=16, Width=16, Depth=5
    pad_width = ((0, 0), (0, 16), (0, 16), (0, 5))

    # mode="constant" pads with a constant value (0) outside the boundaries of the image.
    channels_data = np.pad(channels_data, pad_width, mode="constant", constant_values=0)

    # Create a new NIfTI image with the normalized and padded data, using the same affine and header.
    channels_img = nib.Nifti1Image(
        channels_data.astype(np.float32), img_affine, header=img_header
    )

    # Save the processed image for the current patient
    nib.save(
        channels_img,
        os.path.join(
            processed_data_path,
            f"BraTS2021_{id}",
            f"BraTS2021_{id}.nii.gz",
        ),
    )

    # Load the label image for the patient
    label = nib.load(
        os.path.join(
            raw_data_path,
            f"BraTS2021_{id}",
            f"BraTS2021_{id}_seg.nii.gz",
        )
    )
    lbl_header = label.header
    lbl_affine = label.affine
    lbl_data = label.get_fdata().astype(np.uint8)

    # This ensures classes are [0, 1, 2, 3] instead of [0, 1, 2, 4]
    lbl_data[lbl_data == 4] = 3

    # Expand the dimensions of the label data to add a channel dimension
    lbl_data = np.expand_dims(lbl_data, axis=0)
    # Pad the label data with the same padding as the images to ensure they have the same shape
    lbl_data = np.pad(lbl_data, pad_width, mode="constant", constant_values=0)

    # Create a new NIfTI image for the label with the same affine and header and save it
    ohc_label = nib.Nifti1Image(
        lbl_data.astype(np.uint8), lbl_affine, header=lbl_header
    )
    nib.save(
        ohc_label,
        os.path.join(
            processed_data_path, f"BraTS2021_{id}", f"BraTS2021_{id}_seg.nii.gz"
        ),
    )

In [ ]:
# Make a list of patient IDs by extracting the last part of the directory names in the raw data directory and sort them
pt_ids = [d.split("_")[-1] for d in raw_data_dirs]
pt_ids.sort()

# If all raw data has already been processed, skip the preprocessing step
if len(raw_data_dirs) != len(processed_data_dirs):
    # Ask if the user wants to recalculate the intensity statistics
    # If No, use the existing intensity statistics which were calculated from a previous run
    # If Yes, collect the intensity values from the raw images for all patients
    print("Do you want to recalculate the intensity statistics? (y/n)")
    if input().lower() != "y":
        print("Using existing intensity statistics.")
        intensity_stats = {
            "min": {"t1": 60.00, "t1ce": 47.00, "t2": 115.00, "flair": 115.00},
            "max": {"t1": 6483.00, "t1ce": 7956.00, "t2": 12371.00, "flair": 13702.00},
            "mean": {"t1": 862.64, "t1ce": 1553.63, "t2": 1901.65, "flair": 1080.03},
            "std": {"t1": 1807.43, "t1ce": 16210.56, "t2": 39182.12, "flair": 4100.89},
        }
    else:
        print("Collecting intensity values for normalization...")
        intensity_start = time.time()
        intensity_stats = collect_intensities(pt_ids)
        intensity_end = time.time()
        print(
            f"Intensities collected in {intensity_end - intensity_start:.2f} seconds."
        )

    print("Intensity statistics:")
    print("Min:", intensity_stats["min"])
    print("Max:", intensity_stats["max"])
    print("Mean:", intensity_stats["mean"])
    print("Std:", intensity_stats["std"])

    # Preprocess the images for all patients
    print("Processing raw data...")
    preprocessing_start = time.time()
    preprocess_images(pt_ids)
    preprocessing_end = time.time()
    print(
        f"Image preprocessing completed in {preprocessing_end - preprocessing_start:.2f} seconds."
    )
else:
    print("Preprocessing skipped. All data is already processed.")

### Data Augmentation


In [ ]:
def apply_zoom(image, label, chance):

    if np.random.rand() < chance:
        orig_shape = image.shape[1:]

        # Select a random zoom factor between 1.0 and 1.4
        zoom_factor = np.random.uniform(1.0, 1.4)

        # Only zoom the spatial dimensions (D, H, W) and not the channel dimension (C)
        zoom_tuple = (1.0, zoom_factor, zoom_factor, zoom_factor)

        # Apply Zoom
        # order=3 (cubic) for smooth MRI intensity interpolation and mode="nearest" to prevent introducing new intensity values at the borders
        image = scipy.ndimage.zoom(image, zoom_tuple, order=3, mode="nearest")
        # order=0 (nearest) to prevent creating fake classes in labels
        label = scipy.ndimage.zoom(label, zoom_tuple, order=0, mode="nearest")

        # Crop back to original size.
        _, d, h, w = image.shape
        cd, ch, cw = orig_shape

        z_start = (d - cd) // 2
        y_start = (h - ch) // 2
        x_start = (w - cw) // 2

        image = image[
            :, z_start : z_start + cd, y_start : y_start + ch, x_start : x_start + cw
        ]
        label = label[
            :, z_start : z_start + cd, y_start : y_start + ch, x_start : x_start + cw
        ]

    return image, label


def apply_flips(image, label, chance):
    # Axes for D, H, W (skipping Channel at index 0)
    axes = [1, 2, 3]

    for axis in axes:
        if np.random.rand() < chance:
            image = np.flip(image, axis=axis)
            label = np.flip(label, axis=axis)

    # np.flip can result in non-contiguous arrays which can cause issues with some libraries
    # np.ascontiguousarray makes a copy of the array with contiguous memory layout.
    image = np.ascontiguousarray(image)
    label = np.ascontiguousarray(label)

    return image, label


def apply_gaussian_noise(image, chance):
    if np.random.rand() < chance:
        #  Select a random standard deviation between 0.0 and 0.33
        std = np.random.uniform(0.0, 0.33)
        # Generate Gaussian noise with mean 0 and the selected standard deviation
        noise = np.random.normal(0, std, image.shape)
        # Apply noise
        image = image + noise
    return image.astype(np.float32)


def apply_gaussian_blur(image, chance):
    if np.random.rand() < chance:
        # Select a random sigma value between 0.5 and 1.5 for the Gaussian blur
        sigma = np.random.uniform(0.5, 1.5)
        # Apply Blur
        image = scipy.ndimage.gaussian_filter(image, sigma=(0, sigma, sigma, sigma))
    return image


def apply_brightness(image, chance):
    if np.random.rand() < chance:
        # Select a random brightness factor between 0.7 and 1.3
        factor = np.random.uniform(0.7, 1.3)
        # Apply brightness adjustment
        image = image * factor
    return image


def apply_contrast(image, chance):
    if np.random.rand() < chance:
        # Select a random contrast factor between 0.65 and 1.5
        factor = np.random.uniform(0.65, 1.5)
        img_min, img_max = image.min(), image.max()
        # Apply contrast adjustment
        image = image * factor
        # Clip back to original min/max values
        image = np.clip(image, img_min, img_max)
    return image

In [ ]:
def apply_augmentation(image, label, isTraining=True):
    # 15% chance to apply each augmentation independently
    chance = 0.15
    if isTraining:
        image, label = apply_zoom(image, label, chance)
        image, label = apply_flips(image, label, chance)
        image = apply_gaussian_noise(image, chance)
        image = apply_gaussian_blur(image, chance)
        image = apply_brightness(image, chance)
        image = apply_contrast(image, chance)

    return image, label

### Loading Dataset


In [ ]:
class BraTSDataset(torch.utils.data.Dataset):
    def __init__(self, data_path, pt_ids, isTraining=True):
        self.data_path = data_path
        self.pt_ids = pt_ids
        self.isTraining = isTraining

    # Required: PyTorch needs to know how many total patients there are
    def __len__(self):
        return len(self.pt_ids)

    # Required: PyTorch needs to know how to get the next item
    def __getitem__(self, idx):
        pt_id = self.pt_ids[idx]

        img_path = os.path.join(
            self.data_path, f"BraTS2021_{pt_id}", f"BraTS2021_{pt_id}.nii.gz"
        )
        lbl_path = os.path.join(
            self.data_path, f"BraTS2021_{pt_id}", f"BraTS2021_{pt_id}_seg.nii.gz"
        )

        image = nib.load(img_path).get_fdata().astype(np.float32)
        label = nib.load(lbl_path).get_fdata().astype(np.uint8)

        # Apply data augmentation to the image and label if in training mode
        image, label = apply_augmentation(image, label, self.isTraining)

        # Convert from numpy arrays to PyTorch tensors
        image, label = torch.from_numpy(image), torch.from_numpy(label)

        return image, label

In [ ]:
# Split the patient IDs into training and testing sets using an 80-20 split.
train_ids, test_ids = sklearn.model_selection.train_test_split(
    pt_ids, test_size=0.2, random_state=123
)

In [ ]:
training_dataset = BraTSDataset(processed_data_path, train_ids, isTraining=True)
train_loader = torch.utils.data.DataLoader(training_dataset, batch_size=1, shuffle=True)

val_dataset = BraTSDataset(processed_data_path, test_ids, isTraining=False)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=False)

## Task 3: Implementing the nnU-Net Model


# 3D nnU-Net Architecture Description

---

## 1. The Building Block: `ConvBlock`

The `ConvBlock` is the fundamental unit repeated throughout both the encoder and decoder. It consists of a double convolution sequence:
`Conv3d -> InstanceNorm3d -> LeakyReLU -> Conv3d -> InstanceNorm3d -> LeakyReLU`

**Design Choices & Reasoning:**

- **3D Convolutions (`nn.Conv3d`):** Medical images like MRIs are volumetric (Depth, Height, Width). 3D convolutions are necessary to capture spatial context across all three dimensions simultaneously, which 2D convolutions would miss.
- **`kernel_size=3, padding=1`:** This standard configuration ensures that the convolutional operation itself does not shrink the spatial dimensions of the feature maps, allowing depth/resolution changes to be explicitly controlled by strides.
- **`bias=False`:** Because the convolution is immediately followed by an Instance Normalization layer (which introduces its own bias/shift parameters), adding a bias to the convolution is redundant and wastes memory.
- **Instance Normalization (`nn.InstanceNorm3d`):** Unlike 2D image tasks that use Batch Normalization, 3D medical imaging models consume massive amounts of VRAM, forcing batch sizes to be very small (often 1 or 2). Batch Normalization performs poorly with small batch sizes. Instance Normalization normalizes across each channel of each individual sample, making it highly stable regardless of batch size.
- **Leaky ReLU (`nn.LeakyReLU`):** Replaces the standard ReLU to prevent the "dying neuron" problem. By allowing a small, non-zero gradient when the input is negative, it helps the network continue learning even when weights fall below zero.

---

## 2. The Encoder (Downsampling Path)

The encoder extracts hierarchical features from the input MRI. As the network gets deeper, it reduces the spatial resolution while increasing the number of feature channels (from 4 to 320).

**Design Choices & Reasoning:**

- **Input Channels (4):** Designed to accept the 4 standard BraTS MRI modalities stacked together: T1, T1ce (contrast-enhanced), T2, and FLAIR.
- **Strided Convolutions for Downsampling:** Notice that starting from `enc2`, the `ConvBlock` is passed a `stride=2`. Instead of using separate `MaxPool3d` layers, the model uses strided convolutions to halve the spatial dimensions. This allows the network to _learn_ the optimal pooling operation rather than relying on a fixed mathematical operation.
- **Memory-Efficient Channel Scaling:** A standard U-Net doubles channels at every level (32 $\rightarrow$ 64 $\rightarrow$ 128 $\rightarrow$ 256 $\rightarrow$ 512 $\rightarrow$ 1024). Because 3D data is so large, pushing channels to 512 or 1024 would crash most consumer GPUs. This model intelligently caps the channels at **320** at Level 5 and the Bottleneck to maintain high-level feature extraction without exceeding VRAM limits.

---

## 3. The Bottleneck (Level 6)

The bottleneck (`self.bottleneck = ConvBlock(320, 320, 2)`) sits at the very bottom of the "U".

**Design Choices & Reasoning:**

- **Deepest Semantic Representation:** At this stage, the spatial dimensions are highly compressed, but the channels are rich with semantic context. The network knows "what" it is looking at (e.g., tumor core vs. edema) but has lost the precise spatial boundaries of "where" it is.

---

## 4. The Decoder (Upsampling Path)

The decoder reconstructs the spatial resolution to match the original image size, transforming the deep semantic features back into voxel-level predictions.

**Design Choices & Reasoning:**

- **Transposed Convolutions (`nn.ConvTranspose3d`):** Layers like `self.up5` use `stride=2` and `kernel_size=2` to double the spatial dimensions (Depth, Height, Width) of the feature maps, physically enlarging the tensor.
- **Skip Connections (`torch.cat([..., ...], dim=1)`):** This is the most crucial part of the U-Net. The upsampled features from the decoder are concatenated with the high-resolution features saved from the corresponding level of the encoder.
- _Why?_ The encoder features contain the sharp, fine-grained spatial boundaries (the "where") that were lost during downsampling. The decoder features contain the deep context (the "what"). Concatenating them gives the subsequent `ConvBlock` the best of both worlds, allowing it to draw highly precise segmentation masks.

- **Channel Compression:** When the 320 upsampled channels are concatenated with the 320 skip-connection channels in `dec5`, the input tensor becomes 640 channels deep. `dec5` immediately compresses this back down to 320. This pattern continues (512 $\rightarrow$ 256, etc.) as the network builds its way back up.

---

## 5. Final Segmentation Head

`self.final_conv = torch.nn.Conv3d(32, out_channels, kernel_size=1)`

**Design Choices & Reasoning:**

- **1x1x1 Convolution:** A 1x1 convolution acts as a voxel-wise linear classifier. It doesn't look at neighboring voxels; instead, it looks _across_ the 32 remaining feature channels and collapses them down to the final `out_channels=4`.
- **Output Channels (4):** This outputs raw logits for the 4 expected classes in the BraTS dataset (typically: 0=Background, 1=Necrotic Tumor Core, 2=Peritumoral Edema, 3=Enhancing Tumor). These logits would then be passed through a Softmax or Sigmoid function depending on the specific loss function (e.g., Cross-Entropy or Dice Loss) used during training.


In [ ]:
class ConvBlock(torch.nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = torch.nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False,
            stride=stride,
        )
        self.norm1 = torch.nn.InstanceNorm3d(out_channels)
        self.relu1 = torch.nn.LeakyReLU(inplace=True)

        self.conv2 = torch.nn.Conv3d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.norm2 = torch.nn.InstanceNorm3d(out_channels)
        self.relu2 = torch.nn.LeakyReLU(inplace=True)

    def forward(self, x):
        return self.relu2(self.norm2(self.conv2(self.relu1(self.norm1(self.conv1(x))))))


class nnUNet(torch.nn.Module):
    def __init__(self):
        super().__init__()

        in_channels = 4  # T1, T1ce, T2, FLAIR
        out_channels = 4  # Background, Necrotic Core, Edema, Enhancing Tumor

        # --- ENCODER ---
        # Encoder Level 1
        self.enc1 = ConvBlock(in_channels, 32)
        # Encoder Level 2
        self.enc2 = ConvBlock(32, 64, 2)
        # Encoder Level 3
        self.enc3 = ConvBlock(64, 128, 2)
        # Encoder Level 4
        self.enc4 = ConvBlock(128, 256, 2)
        # Encoder Level 5
        self.enc5 = ConvBlock(256, 320, 2)
        # Bottleneck (Level 6)
        self.bottleneck = ConvBlock(320, 320, 2)

        # --- DECODER ---
        # Decoder Level 5
        self.up5 = torch.nn.ConvTranspose3d(320, 320, kernel_size=2, stride=2)
        self.dec5 = ConvBlock(640, 320)
        # Decoder Level 4
        self.up4 = torch.nn.ConvTranspose3d(320, 256, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(512, 256)
        # Decoder Level 3
        self.up3 = torch.nn.ConvTranspose3d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(256, 128)
        # Decoder Level 2
        self.up2 = torch.nn.ConvTranspose3d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(128, 64)
        # Decoder Level 1
        self.up1 = torch.nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(64, 32)
        # Decoder Level 0 (Final Convolution)
        self.final_conv = torch.nn.Conv3d(32, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        # Bottleneck
        b = self.bottleneck(e5)

        # Decoder
        d5 = self.dec5(torch.cat([self.up5(b), e5], dim=1))
        d4 = self.dec4(torch.cat([self.up4(d5), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        # Final Convolution
        f = self.final_conv(d1)

        return f

## Training


### Evaluation Metrics


In [ ]:
def compute_metrics(predictions, targets):

    # Convert logits to class predictions
    preds_cls = torch.argmax(predictions, dim=1)

    # Ensure targets are the same shape as predictions (remove channel dim if present)
    targets_cls = targets.squeeze(1) if targets.ndim == 5 else targets

    dice, iou, sens, spec, prec = 0.0, 0.0, 0.0, 0.0, 0.0

    # Calculate for tumor classes only (1 to 3)
    classes_to_eval = range(1, 4)
    for c in classes_to_eval:
        p = preds_cls == c
        t = targets_cls == c

        # True Positives (TP): Both prediction and target are the class
        tp = (p & t).sum().float()
        # False Positives (FP): Prediction is the class but target is not
        fp = (p & ~t).sum().float()
        # True Negatives (TN): Both prediction and target are not the class
        tn = (~p & ~t).sum().float()
        # False Negatives (FN): Target is the class but prediction is not
        fn = (~p & t).sum().float()

        # DICE Score
        dice += (2 * tp) / (2 * tp + fp + fn + math.eps)
        # IoU Score (Intersection over Union)
        iou += tp / (tp + fp + fn + math.eps)
        # Sensitivity (Recall): How many actual positives were correctly identified
        sens += tp / (tp + fn + math.eps)
        # Specificity: How many actual negatives were correctly identified
        spec += tn / (tn + fp + math.eps)
        # Precision: How many predicted positives were actually positive
        prec += tp / (tp + fp + math.eps)

    num_eval_classes = len(classes_to_eval)
    return (
        (dice / num_eval_classes).item(),
        (iou / num_eval_classes).item(),
        (sens / num_eval_classes).item(),
        (spec / num_eval_classes).item(),
        (prec / num_eval_classes).item(),
    )

### Model Training


In [ ]:
def setup_training(model, lr):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")
    model = model.to(device)

    # AdamW Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    # Dice + Cross-Entropy Loss
    criterion = monai.losses.DiceCELoss(to_onehot_y=True, softmax=True, batch=True)
    # Cosine Annealing Learning Rate Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    # Mixed Precision Gradient Scaler for faster training on GPU
    scaler = torch.amp.GradScaler("cuda")

    return device, model, optimizer, criterion, scheduler, scaler

In [ ]:
def load_training_state(
    model,
    optimizer,
    scheduler,
    device,
    json_log_path=".training/training_metrics.json",
    resume_checkpoint=".training/model.pth",
):
    metrics_history = []
    start_epoch = 0
    best_val_loss = float("inf")
    epochs_no_improve = 0

    # Load JSON Metrics
    if os.path.exists(json_log_path):
        print(f"=> Found existing metrics log '{json_log_path}'. Loading data...")
        with open(json_log_path, "r") as f:
            metrics_history = json.load(f)
    else:
        print(f"=> No existing metrics log found. Will create '{json_log_path}'.")

    # Load Checkpoint
    if os.path.isfile(resume_checkpoint):
        print(f"=> Loading checkpoint '{resume_checkpoint}'...")
        checkpoint = torch.load(resume_checkpoint, map_location=device)

        start_epoch = checkpoint["epoch"] + 1
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        best_val_loss = checkpoint["best_val_loss"]
        epochs_no_improve = checkpoint["epochs_no_improve"]

        print(
            f"=> Resuming training from epoch {start_epoch + 1} (Best Val Loss: {best_val_loss:.4f})"
        )
    else:
        print(
            f"=> Warning: Checkpoint '{resume_checkpoint}' not found. Starting from scratch."
        )

    return start_epoch, best_val_loss, epochs_no_improve, metrics_history

In [ ]:
def train_epoch(
    model, train_loader, optimizer, criterion, scaler, device, epoch, num_epochs
):

    model.train()
    train_loss = 0.0
    train_pbar = tqdm.tqdm(
        train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", leave=False
    )

    for images, labels in train_pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    return train_loss / len(train_loader)


def validate_epoch(model, val_loader, criterion, device, epoch, num_epochs):

    model.eval()
    val_loss = 0.0
    epoch_metrics = {"dice": 0.0, "iou": 0.0, "sens": 0.0, "spec": 0.0, "prec": 0.0}

    val_pbar = tqdm.tqdm(
        val_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Val]", leave=False
    )

    with torch.no_grad():
        for images, labels in val_pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            b_dice, b_iou, b_sens, b_spec, b_prec = compute_metrics(outputs, labels)
            epoch_metrics["dice"] += b_dice
            epoch_metrics["iou"] += b_iou
            epoch_metrics["sens"] += b_sens
            epoch_metrics["spec"] += b_spec
            epoch_metrics["prec"] += b_prec

            val_pbar.set_postfix(
                {
                    "loss": f"{loss.item():.4f}",
                    "dice": f"{b_dice:.4f}",
                    "iou": f"{b_iou:.4f}",
                }
            )

    num_batches = len(val_loader)
    avg_val_loss = val_loss / num_batches
    avg_metrics = {k: v / num_batches for k, v in epoch_metrics.items()}

    return avg_val_loss, avg_metrics

In [ ]:
def save_epoch_results(
    epoch,
    model,
    optimizer,
    scheduler,
    avg_train_loss,
    avg_val_loss,
    avg_metrics,
    best_val_loss,
    epochs_no_improve,
    metrics_history,
    json_log_path=".training/training_metrics.json",
):

    # 1. Save to JSON
    epoch_data = {
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "val_dice": avg_metrics["dice"],
        "val_iou": avg_metrics["iou"],
        "val_sensitivity": avg_metrics["sens"],
        "val_specificity": avg_metrics["spec"],
        "val_precision": avg_metrics["prec"],
        "learning_rate": scheduler.get_last_lr()[0],
    }
    metrics_history.append(epoch_data)
    with open(json_log_path, "w") as f:
        json.dump(metrics_history, f, indent=4)

    # 2. Checkpointing logic
    if avg_val_loss < best_val_loss:
        print(
            f"  -> Validation loss improved from {best_val_loss:.4f} to {avg_val_loss:.4f}. Saving best model..."
        )
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), f".training/model_{epoch + 1}.pth")
    else:
        epochs_no_improve += 1
        print(
            f"  -> No improvement in validation loss for {epochs_no_improve} epoch(s)."
        )

    checkpoint_state = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_loss": best_val_loss,
        "epochs_no_improve": epochs_no_improve,
    }
    torch.save(checkpoint_state, ".training/model_latest.pth")

    return best_val_loss, epochs_no_improve

In [ ]:
model = nnUNet()

NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
PATIENCE = 5

# 1. Setup
device, model, optimizer, criterion, scheduler, scaler = setup_training(
    model, LEARNING_RATE
)

# 2. Load State
start_epoch, best_val_loss, epochs_no_improve, metrics_history = load_training_state(
    model, optimizer, scheduler, device
)

print("Do you want to start/resume training? (y/n)")
if input().lower() == "y":
    print(f"Starting training for {NUM_EPOCHS} epochs...")

    for epoch in range(start_epoch, NUM_EPOCHS):
        # Train & Validate
        avg_train_loss = train_epoch(
            model, train_loader, optimizer, criterion, scaler, device, epoch, NUM_EPOCHS
        )
        avg_val_loss, avg_metrics = validate_epoch(
            model, val_loader, criterion, device, epoch, NUM_EPOCHS
        )

        scheduler.step()

        # Print Output
        print(
            f"Epoch {epoch + 1}/{NUM_EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}"
        )
        print(
            f"           -> Dice: {avg_metrics['dice']:.4f} | IoU: {avg_metrics['iou']:.4f} | Sens: {avg_metrics['sens']:.4f} | Spec: {avg_metrics['spec']:.4f}"
        )

        # Save Checkpoints and Metrics
        best_val_loss, epochs_no_improve = save_epoch_results(
            epoch,
            model,
            optimizer,
            scheduler,
            avg_train_loss,
            avg_val_loss,
            avg_metrics,
            best_val_loss,
            epochs_no_improve,
            metrics_history,
        )

        # Early Stopping
        if epochs_no_improve >= PATIENCE:
            print(
                f"\nEarly stopping triggered! Validation loss hasn't improved in {PATIENCE} epochs."
            )
            break

    print("Training completed.")
else:
    print("Training skipped.")

In [ ]:
def plot_training_metrics(json_path=".training/training_metrics.json"):
    # 1. Load the metrics from the JSON file
    try:
        with open(json_path, "r") as f:
            metrics = json.load(f)
    except FileNotFoundError:
        print(
            f"Error: Could not find '{json_path}'. Make sure your model has started training and saved the file."
        )
        return

    # 2. Extract lists of parameters
    epochs = [m["epoch"] for m in metrics]
    train_loss = [m["train_loss"] for m in metrics]
    val_loss = [m["val_loss"] for m in metrics]
    val_dice = [m["val_dice"] for m in metrics]
    val_iou = [m["val_iou"] for m in metrics]
    val_sens = [m["val_sensitivity"] for m in metrics]
    val_spec = [m["val_specificity"] for m in metrics]
    val_prec = [m["val_precision"] for m in metrics]
    lrs = [m["learning_rate"] for m in metrics]

    # 3. Create a 2x2 grid of plots
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))

    # Graph 1: Train vs Validation Loss
    axs[0, 0].plot(
        epochs, train_loss, label="Train Loss", marker="o", color="blue", linewidth=2
    )
    axs[0, 0].plot(
        epochs, val_loss, label="Val Loss", marker="o", color="orange", linewidth=2
    )
    axs[0, 0].set_title("Training and Validation Loss", fontsize=14, fontweight="bold")
    axs[0, 0].set_xlabel("Epoch", fontsize=12)
    axs[0, 0].set_ylabel("Loss", fontsize=12)
    axs[0, 0].legend(fontsize=11)
    axs[0, 0].grid(True, linestyle="--", alpha=0.7)

    # Graph 2: Dice and IoU Scores
    axs[0, 1].plot(
        epochs, val_dice, label="Val Dice", marker="s", color="green", linewidth=2
    )
    axs[0, 1].plot(
        epochs, val_iou, label="Val IoU", marker="s", color="red", linewidth=2
    )
    axs[0, 1].set_title("Validation Dice Score and IoU", fontsize=14, fontweight="bold")
    axs[0, 1].set_xlabel("Epoch", fontsize=12)
    axs[0, 1].set_ylabel("Score", fontsize=12)
    axs[0, 1].legend(fontsize=11)
    axs[0, 1].grid(True, linestyle="--", alpha=0.7)

    # Graph 3: Sensitivity, Specificity, Precision
    axs[1, 0].plot(
        epochs, val_sens, label="Sensitivity", marker="^", color="purple", linewidth=2
    )
    axs[1, 0].plot(
        epochs, val_spec, label="Specificity", marker="^", color="brown", linewidth=2
    )
    axs[1, 0].plot(
        epochs, val_prec, label="Precision", marker="^", color="cyan", linewidth=2
    )
    axs[1, 0].set_title(
        "Sensitivity, Specificity & Precision", fontsize=14, fontweight="bold"
    )
    axs[1, 0].set_xlabel("Epoch", fontsize=12)
    axs[1, 0].set_ylabel("Score", fontsize=12)
    axs[1, 0].legend(fontsize=11)
    axs[1, 0].grid(True, linestyle="--", alpha=0.7)

    # Graph 4: Learning Rate
    axs[1, 1].plot(
        epochs, lrs, label="Learning Rate", marker="d", color="black", linewidth=2
    )
    axs[1, 1].set_title("Learning Rate Decay", fontsize=14, fontweight="bold")
    axs[1, 1].set_xlabel("Epoch", fontsize=12)
    axs[1, 1].set_ylabel("Learning Rate", fontsize=12)
    axs[1, 1].legend(fontsize=11)
    axs[1, 1].grid(True, linestyle="--", alpha=0.7)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_training_metrics()

In [ ]:
def visualize_random_patient(pt_ids, model, device="cuda"):
    """
    Randomly selects a patient, runs inference, and plots the FLAIR MRI,
    Ground Truth Mask, and Predicted Mask side-by-side.
    """
    # 1. Pick a random patient from the list
    pt_id = random.choice(pt_ids)
    print(f"Visualizing Patient ID: {pt_id}")

    # 3. Load the data using nibabel
    flair_img = (
        nib.load(".raw/BraTS2021_00000/BraTS2021_00000_flair.nii.gz")
        .get_fdata()
        .astype(np.float32)
    )
    t1_img = (
        nib.load(".raw/BraTS2021_00000/BraTS2021_00000_t1.nii.gz")
        .get_fdata()
        .astype(np.float32)
    )
    t1ce_img = (
        nib.load(".raw/BraTS2021_00000/BraTS2021_00000_t1ce.nii.gz")
        .get_fdata()
        .astype(np.float32)
    )
    t2_img = (
        nib.load(".raw/BraTS2021_00000/BraTS2021_00000_t2.nii.gz")
        .get_fdata()
        .astype(np.float32)
    )

    # Load and remap ground truth mask (BraTS class 4 becomes 3)
    true_mask = (
        nib.load(".raw/BraTS2021_00000/BraTS2021_00000_seg.nii.gz")
        .get_fdata()
        .astype(np.uint8)
    )
    true_mask[true_mask == 4] = 3

    # 4. Prepare data for the model
    # Stack the 4 modalities into a shape of (4, H, W, D)
    input_tensor = np.stack([flair_img, t1_img, t1ce_img, t2_img], axis=0)

    # Simple normalization (z-score) per channel
    for i in range(4):
        input_tensor[i] = (input_tensor[i] - np.mean(input_tensor[i])) / (
            np.std(input_tensor[i]) + 1e-8
        )

    # Convert to PyTorch tensor, add batch dimension -> (1, 4, H, W, D), and move to GPU
    input_tensor = (
        torch.tensor(input_tensor, dtype=torch.float32).unsqueeze(0).to(device)
    )

    # 5. Generate Model Prediction
    model.eval()
    with torch.no_grad():
        with torch.autocast(
            device_type=device, dtype=torch.float16
        ):  # Use Mixed Precision for speed
            outputs = model(input_tensor)
            # Get predicted class via argmax and remove batch dimension
            pred_mask = torch.argmax(outputs, dim=1).squeeze(0).cpu().numpy()

    # 6. Select the best 2D slice to visualize (the one with the largest tumor)
    # Sum up all non-zero pixels in the XY plane across the Z-axis
    tumor_area_per_slice = np.sum(true_mask > 0, axis=(0, 1))
    best_slice_idx = np.argmax(tumor_area_per_slice)

    # Fallback to the middle slice if no tumor is present
    if tumor_area_per_slice[best_slice_idx] == 0:
        best_slice_idx = true_mask.shape[2] // 2

    print(f"Largest tumor found at axial slice: {best_slice_idx}")

    # Extract the 2D slices
    flair_slice = flair_img[:, :, best_slice_idx]
    true_mask_slice = true_mask[:, :, best_slice_idx]
    pred_mask_slice = pred_mask[:, :, best_slice_idx]

    # Rotate slices 90 degrees (Nibabel loads BraTS arrays rotated sideways)
    flair_slice = np.rot90(flair_slice)
    true_mask_slice = np.rot90(true_mask_slice)
    pred_mask_slice = np.rot90(pred_mask_slice)

    # Mask out the background (0) so it doesn't colorize empty brain space
    true_mask_slice = np.ma.masked_where(true_mask_slice == 0, true_mask_slice)
    pred_mask_slice = np.ma.masked_where(pred_mask_slice == 0, pred_mask_slice)

    # 7. Plotting
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # A. Original Image
    axes[0].imshow(flair_slice, cmap="gray")
    axes[0].set_title("Input MRI (FLAIR Modality)", fontsize=14)
    axes[0].axis("off")

    # B. Ground Truth Overlay
    axes[1].imshow(flair_slice, cmap="gray")
    axes[1].imshow(true_mask_slice, cmap="Set1", alpha=0.6, interpolation="none")
    axes[1].set_title("Ground Truth Mask", fontsize=14)
    axes[1].axis("off")

    # C. Prediction Overlay
    axes[2].imshow(flair_slice, cmap="gray")
    axes[2].imshow(pred_mask_slice, cmap="Set1", alpha=0.6, interpolation="none")
    axes[2].set_title("nnU-Net Prediction", fontsize=14)
    axes[2].axis("off")

    plt.suptitle(
        f"Segmentation Comparison | Patient: {pt_id} | Slice Z={best_slice_idx}",
        fontsize=18,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_random_patient(pt_ids, model=model, device="cuda")